In [1]:
# --- الخلية 1: التثبيت الذكي (Smart Offline Install) ---
import sys
import os
import glob
import subprocess

print("⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...")

# 1. تحديد المكتبات الخطرة التي لا يجب إعادة تثبيتها لأنها موجودة ومجهزة للـ GPU
FORBIDDEN_PACKAGES = [
    "torch-", "torchvision", "torchaudio",  # نترك نسخ Kaggle الأصلية
    "numpy", "pandas", "opencv", "matplotlib", "scipy", "pillow" # موجودة مسبقاً
]

# 2. العثور على الملفات
all_whls = glob.glob("/kaggle/input/**/*.whl", recursive=True)
print(f"📦 تم العثور على {len(all_whls)} ملف في الداتا سيت.")

success_count = 0
for path in all_whls:
    filename = os.path.basename(path).lower()
    
    # التحقق: هل هذا الملف "ممنوع"؟
    is_forbidden = False
    for forbidden in FORBIDDEN_PACKAGES:
        if forbidden in filename:
            is_forbidden = True
            break
    
    if is_forbidden:
        # print(f"🛡️ تم الحفاظ على نسخة النظام من: {filename}")
        continue

    # التثبيت الآمن للمكتبات المفقودة فقط (مثل YOLO, SMP)
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", 
            path, 
            "--no-deps",
            "--ignore-installed"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        success_count += 1
        # print(f"✅ تم تثبيت: {filename}")
    except Exception as e:
        print(f"⚠️ فشل تثبيت: {filename}")

print(f"✅ تمت العملية! تم تثبيت {success_count} مكتبة إضافية.")
print("⚡ إعدادات الـ Torch الحالية:")
import torch
print(f"   - Torch Version: {torch.__version__}")
print(f"   - CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   - GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ تنبيه خطير: الـ GPU غير مفعل! تأكد من إعدادات Kaggle Sidebar.")

# إضافة مسارات YOLO إذا لزم الأمر
src_dirs = glob.glob("/kaggle/input/**/ultralytics", recursive=True)
for d in src_dirs:
    parent = os.path.dirname(d)
    if parent not in sys.path:
        sys.path.append(parent)

⚙️ جاري تثبيت المكتبات الضرورية فقط (حماية الـ GPU)...
📦 تم العثور على 95 ملف في الداتا سيت.
⚠️ فشل تثبيت: pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.5.1-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: markupsafe-3.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
⚠️ فشل تثبيت: triton-3.0.0-1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
⚠️ فشل تثبيت: pyyaml-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl
⚠️ فشل تثبيت: markupsafe-2.1.5-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: charset_normalizer-3.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
⚠️ فشل تثبيت: contourpy-1.2.1-cp310-cp310-manylinux_2_17_x

In [2]:
# --- Cell 2: Offline install/import fix (NO internet) ---
import sys, os, glob, subprocess, site
from importlib import reload

print("🔧 Cell 2: Offline install/import fix (no internet).")

# تعطيل أي شيء ممكن يحاول يتصل بالنت
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def ensure_pkg(import_name: str, wheel_hint: str):
    """Try import; if missing install from /kaggle/input wheels offline."""
    try:
        __import__(import_name)
        print(f"✅ Already available: {import_name}")
        return True
    except Exception:
        wheels = glob.glob(f"/kaggle/input/**/*{wheel_hint}*.whl", recursive=True)
        if not wheels:
            print(f"❌ Missing wheel for: {import_name} (hint={wheel_hint})")
            return False
        target = sorted(wheels, key=len)[0]
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", target, "--no-deps", "--ignore-installed", "--quiet"],
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
            )
            reload(site)
            __import__(import_name)
            print(f"✅ Installed offline: {import_name}")
            return True
        except Exception as e:
            print(f"⚠️ Offline install failed for {import_name}: {e}")
            return False

# نضمن فقط المكتبات التي نحتاجها (بدون لمس torch)
ensure_pkg("timm", "timm")
ensure_pkg("segmentation_models_pytorch", "segmentation_models_pytorch")
ensure_pkg("ultralytics", "ultralytics")

# محرك parquet (عادة موجود في Kaggle)
try:
    import pyarrow  # noqa
    print("✅ pyarrow available (parquet OK)")
except Exception:
    # إذا مو موجود غالبًا Kaggle يوفره؛ لا نحاول تنزيله من النت
    print("⚠️ pyarrow not found. If parquet read fails later, Kaggle environment may be missing parquet engine.")

import torch
import segmentation_models_pytorch as smp

print("------ Environment Check ------")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("smp:", getattr(smp, "__version__", "unknown"))
print("✅ Cell 2 ready.")


🔧 Cell 2: Offline install/import fix (no internet).
✅ Already available: timm
✅ Installed offline: segmentation_models_pytorch
✅ Already available: ultralytics
✅ pyarrow available (parquet OK)
------ Environment Check ------
torch: 2.6.0+cu124
cuda available: True
gpu: Tesla T4
smp: 0.5.0
✅ Cell 2 ready.


In [3]:
# --- الخلية 3: محرك الرسم الحي (Ultimate Renderer) - [M3: UPDATED] ---
# إعداد قاعدة البيانات (تحميل مرة واحدة فقط)
DB_DIR = "physionet_db"
if not os.path.exists(DB_DIR):
    os.makedirs(DB_DIR)
    print("⬇️ جاري تحميل سجلات PTB-XL الأساسية...")
    try:
        # تحميل عينة كافية لضمان التنوع
        records = wfdb.get_record_list('ptb-xl')
        selected_records = records[:200] 
        wfdb.dl_database('ptb-xl', os.getcwd() + "/" + DB_DIR, selected_records)
        print(f"✅ تم تحميل {len(selected_records)} سجل.")
    except Exception as e:
        print(f"⚠️ تحذير: فشل التحميل ({e})، سيتم استخدام التوليد الاحتياطي.")

class UltimateRenderer:
    def __init__(self, db_dir):
        self.db_dir = db_dir
        self.records = [f.split('.')[0] for f in os.listdir(db_dir) if f.endswith('.dat')] if os.path.exists(db_dir) else []
        
    def get_real_signal(self):
        """سحب إشارة عشوائية من PTB-XL"""
        if not self.records:
            t = np.linspace(0, 4, 2000); return np.sin(2 * np.pi * 5 * t) # Fallback
            
        try:
            rec_name = random.choice(self.records)
            record, meta = wfdb.rdsamp(f"{self.db_dir}/{rec_name}")
            lead_idx = random.randint(0, record.shape[1] - 1)
            signal = record[:, lead_idx]
            signal = np.nan_to_num(signal)
            
            # قص عشوائي (Zoom in/out simulation)
            crop_len = random.randint(1000, 3000)
            if len(signal) > crop_len:
                start = random.randint(0, len(signal) - crop_len)
                return signal[start : start+crop_len]
            return signal
        except:
            return np.random.randn(2000)

    def render_to_memory(self, signal):
        """الرسم بدقة عالية (DPI=200) في الذاكرة مباشرة"""
        # 3. الشبكة المتغيرة (Distractor)
        grid_color = random.choice(['red', '#ffb6c1', '#cfcfcf', '#e0e0e0', 'pink'])
        grid_alpha = random.uniform(0.3, 0.8)
        bg_color = random.choice(['white', '#fffdf5', '#f0f0f0']) 
        
        fig_h, fig_w = 3, 8
        dpi = 200 # حسب الطلب لضمان وضوح الحواف
        
        # --- أ. توليد القناع (Mask Generation) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.plot(signal, color='white', linewidth=3.0) 
        ax.set_ylim(np.min(signal), np.max(signal))
        ax.axis('off')
        fig.patch.set_facecolor('black')
        
        fig.canvas.draw()
        mask = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        mask = mask.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(mask, 10, 255, cv2.THRESH_BINARY)
        plt.close(fig)

        # --- ب. الرسم الأولي (Rendering) ---
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=dpi)
        ax.set_facecolor(bg_color)
        
        # رسم الشبكة
        ax.minorticks_on()
        ax.grid(which='major', linestyle='-', linewidth=0.8, color=grid_color, alpha=grid_alpha)
        ax.grid(which='minor', linestyle=':', linewidth=0.4, color=grid_color, alpha=grid_alpha*0.5)
        
        # رسم الإشارة (محاكاة ألوان الحبر المختلفة)
        line_color = random.choice(['black', 'blue', '#000033']) 
        ax.plot(signal, color=line_color, linewidth=random.uniform(1.0, 1.8))
        
        ax.axis('off')
        ax.set_ylim(np.min(signal), np.max(signal))
        fig.patch.set_facecolor(bg_color)
        
        fig.canvas.draw()
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        # نحتفظ بها BGR هنا لأن OpenCV يفضل ذلك للمعالجة اللاحقة (Distractors)
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR) 
        plt.close(fig)
        
        return img, mask

print("✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).")

⬇️ جاري تحميل سجلات PTB-XL الأساسية...
⚠️ تحذير: فشل التحميل (name 'wfdb' is not defined)، سيتم استخدام التوليد الاحتياطي.
✅ تم تحديث الخلية 3: محرك الرسم الحي جاهز (DPI=200).


In [4]:
# --- Cell 22: Merged Ultra Safe Engine (Parquet + Streaming + Anti-OOM + Anti-Crash) ---
import gc, os, csv, glob, math
import numpy as np
import pandas as pd
import cv2
import torch
from collections import OrderedDict
from tqdm import tqdm
from scipy.signal import savgol_filter, resample, find_peaks, butter, filtfilt

import segmentation_models_pytorch as smp
from ultralytics import YOLO

# ---------- Hard Offline Guards ----------
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.set_grad_enabled(False)

# ---------- Paths / Constants ----------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEST_CSV_PATH = "/kaggle/input/physionet-ecg-image-digitization/test.csv"
SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"
IMAGE_DIR = "/kaggle/input/physionet-ecg-image-digitization"

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
LEAD_TO_IDX = {n: i for i, n in enumerate(LEAD_NAMES)}

print(f"⚡ Device: {DEVICE}")

# ---------- 1) Read Parquet Template (MUST) ----------
print("📦 Reading Parquet template ids...")
template_ids = None
try:
    template = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
    template_ids = template["id"].astype(str).to_numpy()
    del template
    gc.collect()
    print(f"✅ Template rows: {len(template_ids):,}")
except Exception as e:
    # لا نكسر النوتبوك: لكن هذا يعني غالبًا أن بيئة المسابقة غير صحيحة
    print("❌ Failed to read sample_submission.parquet:", e)
    print("⚠️ Emergency dummy template (will not score, but avoids crash).")
    template_ids = np.array([f"00000_{i}_I" for i in range(100)], dtype=object)

# ---------- 2) FS Map (Safe) ----------
fs_map = {}
try:
    if os.path.exists(TEST_CSV_PATH):
        tdf = pd.read_csv(TEST_CSV_PATH, dtype=str)
        cols_lower = {c.lower(): c for c in tdf.columns}
        id_col = next((cols_lower[c] for c in cols_lower if "id" in c), None)
        fs_col = next((cols_lower[c] for c in cols_lower if "fs" in c), None)
        if id_col and fs_col:
            fs_map = dict(zip(tdf[id_col].astype(str), tdf[fs_col].astype(str)))
    print(f"✅ fs_map loaded: {len(fs_map):,} items")
except Exception as e:
    print("⚠️ Failed reading test.csv:", e)
    fs_map = {}

def safe_float(x, default=500.0):
    try:
        v = float(x)
        if not np.isfinite(v) or v <= 0:
            return float(default)
        return v
    except Exception:
        return float(default)

def safe_fs(pid, default=500.0):
    pid_s = str(pid)
    return safe_float(fs_map.get(pid_s, fs_map.get(pid_s[:-2] if pid_s.endswith(".0") else pid_s, default)), default)

# ---------- 3) Index Images (Robust) ----------
print("🗂️ Indexing images...")
image_paths = glob.glob(f"{IMAGE_DIR}/**/*.png", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpg", recursive=True) + glob.glob(f"{IMAGE_DIR}/**/*.jpeg", recursive=True)
path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
print(f"✅ Indexed images: {len(path_map):,}")

def clean_pid(pid):
    s = str(pid).strip()
    return s[:-2] if s.endswith(".0") else s

def get_image_path_safe(pid):
    pid_s = clean_pid(pid)
    if pid_s in path_map:
        return path_map[pid_s]
    try:
        pid_int = str(int(float(pid_s)))
        return path_map.get(pid_int)
    except Exception:
        return None

# ---------- 4) Load Models (Offline-safe) ----------
print("⚙️ Loading models (offline)...")
YOLO_PATH = "/kaggle/input/ecg-final-models/best.pt"
UNET_PATH = "/kaggle/input/ecg-final-models/best_model_phase10.pth"

yolo_model = None
try:
    if os.path.exists(YOLO_PATH):
        yolo_model = YOLO(YOLO_PATH)
        print("✅ YOLO loaded.")
    else:
        print("⚠️ YOLO not found -> YOLO disabled (fallback grid crops only).")
except Exception as e:
    print("⚠️ YOLO load failed -> disabled:", e)
    yolo_model = None

unet_model = None
try:
    unet_model = smp.Unet(
        encoder_name="resnet50", encoder_weights=None,
        in_channels=3, classes=1, decoder_attention_type="scse"
    )
    if os.path.exists(UNET_PATH):
        sd = torch.load(UNET_PATH, map_location=DEVICE)
        # strict=False to avoid crash on minor key mismatches
        unet_model.load_state_dict(sd, strict=False)
        print("✅ UNet loaded.")
    else:
        print("⚠️ UNet not found -> UNet disabled (all zeros).")
        unet_model = None

    if unet_model is not None:
        unet_model.to(DEVICE)
        unet_model.eval()
except Exception as e:
    print("⚠️ UNet init/load failed -> disabled:", e)
    unet_model = None

# ---------- 5) Safe helper functions ----------
def apply_high_pass_filter(data, cutoff=0.5, fs=500.0, order=5):
    try:
        if len(data) < order * 3:
            return data
        nyq = 0.5 * fs
        if nyq <= 0:
            return data
        normal_cutoff = cutoff / nyq
        if not (0 < normal_cutoff < 1):
            return data
        b, a = butter(order, normal_cutoff, btype="high", analog=False)
        return filtfilt(b, a, data)
    except Exception:
        return data

def smart_einthoven_fix(leads_dict):
    try:
        if 'I' in leads_dict and 'II' in leads_dict and 'III' in leads_dict:
            l = min(len(leads_dict['I']), len(leads_dict['II']), len(leads_dict['III']))
            I, II, III = leads_dict['I'][:l], leads_dict['II'][:l], leads_dict['III'][:l]
            residual = (I + III) - II
            leads_dict['II'][:l] += residual / 3.0
            leads_dict['I'][:l]  -= residual / 3.0
            leads_dict['III'][:l]-= residual / 3.0
    except Exception:
        pass
    return leads_dict

def robust_multi_point_calibration(crop, default_val=25.0):
    try:
        if crop is None or crop.size == 0:
            return float(default_val)
        gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
        if gray.size == 0 or np.std(gray) < 3:
            return float(default_val)
        edges_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        edges_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sx = np.sum(np.abs(edges_x), axis=0)
        sy = np.sum(np.abs(edges_y), axis=1)
        peaks_x, _ = find_peaks(sx, distance=10, prominence=max(5.0, np.percentile(sx, 90)*0.05))
        peaks_y, _ = find_peaks(sy, distance=10, prominence=max(5.0, np.percentile(sy, 90)*0.05))
        grid_sizes = []
        if len(peaks_x) > 3:
            dx = np.diff(peaks_x)
            grid_sizes.extend(dx[(dx > 10) & (dx < 120)])
        if len(peaks_y) > 3:
            dy = np.diff(peaks_y)
            grid_sizes.extend(dy[(dy > 10) & (dy < 120)])
        if len(grid_sizes) >= 5:
            return float(np.median(np.array(grid_sizes)))
    except Exception:
        pass
    return float(default_val)

def advanced_perspective_correction(image):
    try:
        if image is None:
            return None
        h, w = image.shape[:2]
        if h < 32 or w < 32:
            return image
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        small = cv2.resize(gray, (max(32, w//4), max(32, h//4)))
        edges = cv2.Canny(small, 50, 150)
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=50,
                                minLineLength=max(10, small.shape[1]//8), maxLineGap=10)
        if lines is None:
            return image
        angles = []
        for l in lines:
            x1, y1, x2, y2 = l[0]
            ang = np.degrees(np.arctan2((y2-y1), (x2-x1)))
            if abs(ang) < 20:
                angles.append(ang)
        if len(angles) >= 5:
            med = float(np.median(angles))
            if 0.5 < abs(med) < 20:
                M = cv2.getRotationMatrix2D((w//2, h//2), med, 1.0)
                return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
    except Exception:
        pass
    return image

def preprocess_remove_grid_rgb(image_rgb):
    try:
        if image_rgb is None:
            return None
        hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
        mask = (cv2.inRange(hsv, (0, 50, 50), (10, 255, 255)) |
                cv2.inRange(hsv, (170, 50, 50), (180, 255, 255)))
        out = image_rgb.copy()
        out[mask > 0] = (255, 255, 255)
        return out
    except Exception:
        return image_rgb

def get_yolo_crops_with_fallback(image, model):
    h, w = image.shape[:2]
    crops = [None] * 12

    # YOLO attempt
    if model is not None:
        try:
            dev = 0 if DEVICE == "cuda" else "cpu"   # ✅ fix: ulralytics expects 0 or "cpu"
            results = model.predict(image, verbose=False, conf=0.15, imgsz=640, device=dev)
            if results and results[0].boxes is not None and len(results[0].boxes) > 0:
                boxes = results[0].boxes.data.detach().cpu().numpy()
                best = {}
                for b in boxes:
                    cls_id = int(b[5])
                    conf = float(b[4])
                    if 0 <= cls_id < 12:
                        if cls_id not in best or conf > best[cls_id][0]:
                            best[cls_id] = (conf, b[:4])

                for cid, (_, xyxy) in best.items():
                    x1, y1, x2, y2 = map(int, xyxy)
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(w, x2), min(h, y2)
                    if x2 > x1 and y2 > y1:
                        crops[cid] = image[y1:y2, x1:x2]

                if sum(c is not None for c in crops) >= 8:
                    return crops
        except Exception:
            pass

    # fallback grid
    rh, cw = h // 4, w // 3
    if rh < 5 or cw < 5:
        return crops
    return [image[r*rh:(r+1)*rh, c*cw:(c+1)*cw] for r in range(4) for c in range(3)]

def safe_savgol(y, W, poly=2):
    # ✅ make window odd and >= poly+2 and <= len(y)
    n = len(y)
    if n < 5:
        return y
    w = min(W, n)
    if w % 2 == 0:
        w -= 1
    w = max(w, poly + 3)
    if w > n:
        w = n if n % 2 == 1 else n - 1
    if w < poly + 3:
        return y
    return savgol_filter(y, w, poly)

def fast_viterbi(prob_map):
    # stable & fast
    try:
        H, W = prob_map.shape
        if H == 0 or W == 0:
            return np.zeros(W, dtype=np.float32)
        path = np.argmax(prob_map, axis=0).astype(np.float32)
        path = safe_savgol(path, 15, 2)
        return (H - path).astype(np.float32)
    except Exception:
        return np.zeros(prob_map.shape[1], dtype=np.float32)

def batch_predict_unet(crops, model):
    if (not crops) or (model is None):
        return [], []
    target_h = 256
    processed, scales_h, widths = [], [], []

    try:
        for img in crops:
            h, w = img.shape[:2]
            if h <= 1 or w <= 1:
                continue

            scale_h = target_h / float(h)
            new_w = int(max(2, w * scale_h))

            # ✅ CRITICAL: clamp width to avoid OOM
            if new_w > 2048:
                new_w = 2048

            img_rz = cv2.resize(img, (new_w, target_h))
            tens = torch.from_numpy(img_rz).permute(2, 0, 1).contiguous().float() / 255.0

            processed.append(tens)
            scales_h.append(scale_h)
            widths.append(new_w)

        if not processed:
            return [], []

        max_w = int(math.ceil(max(widths) / 32) * 32)
        max_w = min(max_w, 3072)  # hard cap

        batch = torch.zeros((len(processed), 3, target_h, max_w), dtype=torch.float32, device=DEVICE)
        for i, t in enumerate(processed):
            w_curr = min(t.shape[2], max_w)
            batch[i, :, :t.shape[1], :w_curr] = t[:, :, :w_curr].to(DEVICE)

        with torch.no_grad():
            p1 = torch.sigmoid(model(batch))
            p2 = torch.sigmoid(model(torch.flip(batch, [3])))
            p = (p1 + torch.flip(p2, [3])) / 2.0

        p = p.detach().cpu().numpy()
        prob_maps = []
        for i in range(len(processed)):
            prob_maps.append(p[i, 0, :target_h, :widths[i]])
        return prob_maps, scales_h

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            return [], []
        raise
    except Exception:
        return [], []

# ---------- 6) Patient Cache + Compute ----------
patient_cache = OrderedDict()
MAX_CACHE_PATIENTS = 12  # safer

def compute_patient_leads(pid_clean_str):
    default_data = np.zeros((12, 5000), dtype=np.float32)
    try:
        path = get_image_path_safe(pid_clean_str)
        if not path:
            return default_data

        img = cv2.imread(path)
        if img is None:
            return default_data
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        fs_val = safe_fs(pid_clean_str, 500.0)

        img = advanced_perspective_correction(img)
        global_grid = robust_multi_point_calibration(img, 25.0)

        crops = get_yolo_crops_with_fallback(img, yolo_model)

        clean_crops, valid_idx = [], []
        for i, crop in enumerate(crops):
            if crop is not None and crop.size > 100:
                clean_crops.append(preprocess_remove_grid_rgb(crop))
                valid_idx.append(i)

        if not clean_crops:
            return default_data

        prob_maps, scales_h = batch_predict_unet(clean_crops, unet_model)
        if not prob_maps:
            return default_data

        leads_dict = {}
        for j, real_idx in enumerate(valid_idx):
            if j >= len(prob_maps):
                break
            prob = prob_maps[j]
            scale_h = scales_h[j] if j < len(scales_h) else 1.0
            lname = LEAD_NAMES[real_idx] if 0 <= real_idx < 12 else None
            if lname is None:
                continue

            local_grid = robust_multi_point_calibration(clean_crops[j], default_val=global_grid)
            raw = fast_viterbi(prob)

            g_sc = float(local_grid) * float(scale_h)
            divider = (g_sc * 10.0) if g_sc > 1.0 else 250.0

            sig_mv = (raw - np.median(raw)) / float(divider)
            sig_mv = np.nan_to_num(sig_mv, nan=0.0, posinf=0.0, neginf=0.0)

            if len(sig_mv) > 15:
                sig_mv = safe_savgol(sig_mv, 11, 3)
            if len(sig_mv) > 20:
                sig_mv = apply_high_pass_filter(sig_mv, cutoff=0.5, fs=fs_val)

            if len(sig_mv) > 0:
                sig_rs = resample(sig_mv, 5000).astype(np.float32)
                sig_rs = np.nan_to_num(sig_rs, nan=0.0, posinf=0.0, neginf=0.0)
                leads_dict[lname] = sig_rs

        leads_dict = smart_einthoven_fix(leads_dict)

        final = np.zeros((12, 5000), dtype=np.float32)
        for i, l in enumerate(LEAD_NAMES):
            if l in leads_dict:
                final[i] = leads_dict[l]
        return final

    except Exception:
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        return default_data

# ---------- 7) Streaming Write ----------
print("🚀 Writing submission.csv (Merged Safe Streaming)...")
with open(SUBMISSION_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "value"])

    for row_id in tqdm(template_ids, desc="Processing Rows"):
        try:
            parts = str(row_id).rsplit("_", 2)
            if len(parts) != 3:
                writer.writerow([row_id, "0.0000"])
                continue

            pid, idx_s, lead = parts[0], parts[1], parts[2]
            try:
                idx = int(idx_s)
            except Exception:
                writer.writerow([row_id, "0.0000"])
                continue

            pid_c = clean_pid(pid)

            if pid_c in patient_cache:
                mat = patient_cache[pid_c]
                patient_cache.move_to_end(pid_c)
            else:
                mat = compute_patient_leads(pid_c)
                patient_cache[pid_c] = mat
                if len(patient_cache) > MAX_CACHE_PATIENTS:
                    patient_cache.popitem(last=False)

            val = 0.0
            if 0 <= idx < 5000:
                lead_idx = LEAD_TO_IDX.get(lead, 0)
                v = float(mat[lead_idx][idx])
                if np.isfinite(v):
                    val = v

            writer.writerow([row_id, f"{val:.4f}"])

        except Exception:
            writer.writerow([row_id, "0.0000"])

del patient_cache
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("✅ Done. submission.csv created successfully.")


⚡ Device: cuda
📦 Reading Parquet template ids...
✅ Template rows: 75,000
✅ fs_map loaded: 2 items
🗂️ Indexing images...
✅ Indexed images: 8,795
⚙️ Loading models (offline)...
✅ YOLO loaded.
✅ UNet loaded.
🚀 Writing submission.csv (Merged Safe Streaming)...


Processing Rows: 100%|██████████| 75000/75000 [00:03<00:00, 24668.57it/s]


✅ Done. submission.csv created successfully.


In [5]:
# =========================
# --- الخلية 23: Strict Submission Validator (Must Pass) ---
# =========================
import os
import numpy as np
import pandas as pd

SAMPLE_PARQUET_PATH = "/kaggle/input/physionet-ecg-image-digitization/sample_submission.parquet"
SUBMISSION_FILE = "submission.csv"

print("🔍 Validating submission.csv strictly...")

if not os.path.exists(SUBMISSION_FILE):
    raise FileNotFoundError("❌ submission.csv not found. Run Cell 22 first.")

# اقرأ القالب (ids فقط)
tmpl = pd.read_parquet(SAMPLE_PARQUET_PATH, columns=["id"])
tmpl_ids = tmpl["id"].astype(str).to_numpy()
n_tmpl = len(tmpl_ids)

# اقرأ ملفك
sub = pd.read_csv(SUBMISSION_FILE)

# 1) الأعمدة
assert list(sub.columns) == ["id", "value"], f"❌ Columns mismatch: {sub.columns}"

# 2) عدد الصفوف
assert len(sub) == n_tmpl, f"❌ Row count mismatch: sub={len(sub)} vs template={n_tmpl}"

# 3) عدم وجود NaN
assert sub["id"].isna().sum() == 0, "❌ Found NaN in id"
assert sub["value"].isna().sum() == 0, "❌ Found NaN in value"

# 4) تحويل value إلى float والتأكد finite
vals = pd.to_numeric(sub["value"], errors="coerce").to_numpy()
assert np.isfinite(vals).all(), "❌ Found non-finite values (NaN/inf) in value"

# 5) مطابقة أول وآخر ID (كفاية لاكتشاف تغيير ترتيب/تنظيف خاطئ)
sub_ids = sub["id"].astype(str).to_numpy()
assert sub_ids[0] == tmpl_ids[0], f"❌ First ID mismatch: {sub_ids[0]} != {tmpl_ids[0]}"
assert sub_ids[-1] == tmpl_ids[-1], f"❌ Last ID mismatch: {sub_ids[-1]} != {tmpl_ids[-1]}"

# 6) فحص عينة عشوائية للمطابقة (بدون مقارنة كل شيء لتوفير وقت)
idxs = np.linspace(0, n_tmpl-1, num=min(20, n_tmpl), dtype=int)
for i in idxs:
    if sub_ids[i] != tmpl_ids[i]:
        raise AssertionError(f"❌ ID mismatch at row {i}: {sub_ids[i]} != {tmpl_ids[i]}")

size_mb = os.path.getsize(SUBMISSION_FILE) / (1024*1024)
print("✅ All checks passed.")
print(f"📦 submission.csv size: {size_mb:.2f} MB")
print("🎉 Ready to Submit.")


🔍 Validating submission.csv strictly...
✅ All checks passed.
📦 submission.csv size: 1.96 MB
🎉 Ready to Submit.
